**Archived legacy method.** Set `YEAR` and, only when intentionally writing disposable outputs, set `SAVE = True`. Outputs go to `notebooks/outputs/<YEAR>/`; this notebook does not publish canonical pipeline data.

In [ ]:
# Legacy notebook controls — edit only these values
YEAR = 2026
SAVE = False

from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_ROOT = REPO_ROOT / "data" / "raw"
PARTITION_ROOT = REPO_ROOT / "data" / "partitions"
PRODUCT_ROOT = REPO_ROOT / "data" / "products"
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "outputs" / str(YEAR)
GEODATA_OUTPUT_ROOT = OUTPUT_ROOT / "geodata"

DEPARTMENTS_PATH = RAW_ROOT / "demographics" / "departments.csv"
DEPARTMENT_STATS_PATH = RAW_ROOT / "demographics" / "departmental_stats_2023.csv"
ARRONDISSEMENT_STATS_PATH = RAW_ROOT / "demographics" / "arrondissement_stats_2023.csv"
DEPARTMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "departments.geojson"
REGIONS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "regions.geojson"
ARRONDISSEMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "arrondissements-avec-outre-mer.geojson"
PARIS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "paris_arrondissements.geojson"
MONACO_GEOMETRY_PATH = RAW_ROOT / "geodata" / "monaco.geojson"


def annual_restaurant_product(year):
    candidates = [
        PRODUCT_ROOT / "france" / str(year) / "all_restaurants(arrondissements).csv",
        PRODUCT_ROOT / "france" / f"all_restaurants(arrondissements)_{year % 100:02d}.csv",
        REPO_ROOT / "Years" / str(year) / "data" / "France" / "all_restaurants(arrondissements).csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"No accepted annual restaurant product for {year}: {candidates}")


def save_csv(frame, filename):
    target = OUTPUT_ROOT / filename
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=False)
    print(f"Wrote {target}")


def save_geojson(frame, filename):
    target = GEODATA_OUTPUT_ROOT / filename
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_file(target, driver="GeoJSON")
    print(f"Wrote {target}")


# Monaco restaurants

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
monaco = pd.read_csv(PARTITION_ROOT / "monaco" / f"monaco_{YEAR}.csv")
monaco

In [ ]:
# We drop 'city'
monaco.drop(['city', 'country'], axis=1, inplace=True)

In [ ]:
print(f"Shape of DataFrame: {monaco.shape}")
print(f"Columns:\n{monaco.columns.unique().tolist()}")

In [ ]:
print(monaco.info())

----
&nbsp;
### Restaurant Locations Visualisation

In [ ]:
monaco.plot(kind='scatter', x='longitude', y='latitude', alpha=0.2)

----
&nbsp;
### Amendments to the address and location formatting

In [ ]:
# Check for addresses without a comma or with more than 3 commas (indicating more than 4 fields)
incorrect_format = monaco[~monaco['address'].str.count(',').isin([1, 2, 3])]

In [ ]:
# Print out number addresses that don't match the expected format
if not incorrect_format.empty:
    print(f"Addresses with unexpected format: {len(incorrect_format['address'])}")

In [ ]:
def process_address(addr):
    # Split the address by commas and strip whitespace
    parts = [part.strip() for part in addr.split(',')]

    # If there are not enough parts, just return the original
    if len(parts) < 4:
        return addr, None, None

    # Extract the postal code and city
    postal_code = parts[-2]
    location = parts[-3]

    # Form the main address by joining the remaining parts (excluding postal code, city, and country)
    main_address = ', '.join(parts[:-3])

    return main_address, location, postal_code

In [ ]:
# Apply the function to the DataFrame
addresses, locations, postal_codes = zip(*monaco['address'].apply(process_address))

monaco['address'] = addresses
monaco['location'] = [f"{location}, {postal_code}" for location, postal_code in zip(locations, postal_codes)]
monaco['arrondissement'] = 'Monaco'
monaco['department_num'] = '98'
monaco['department'] = 'Monaco'
monaco['capital'] = 'Monaco'
monaco['region'] = "Provence-Alpes-Côte d'Azur"

In [ ]:
monaco = monaco[['name', 'address', 'location', 'arrondissement', 'department_num', 'department', 'capital', 'region', 'price', 'cuisine', 'url', 'award', 'greenstar', 'stars', 'longitude', 'latitude']]
monaco.head()

In [ ]:
monaco['department_num'] = monaco['department_num'].astype(str)
monaco.info()

We export `monaco` as `monaco_restaurants.csv`

In [ ]:
# Export the DataFrame to a .csv file
save_csv(monaco, "monaco_restaurants.csv")

----
&nbsp;
### Grouping restaurants based on the number of Michelin stars.

In [ ]:
# We create a copy
monaco_copy = monaco.copy()

# Create dummy variables for each category of 'star'
monaco_copy['green_stars'] = monaco_copy['greenstar'].apply(lambda x: 1 if x == 1 else 0)
monaco_copy['selected'] = monaco_copy['stars'].apply(lambda x: 1 if x == 0.25 else 0)
monaco_copy['bib_gourmand'] = monaco_copy['stars'].apply(lambda x: 1 if x == 0.5 else 0)
monaco_copy['1_star'] = monaco_copy['stars'].apply(lambda x: 1 if x == 1.0 else 0)
monaco_copy['2_star'] = monaco_copy['stars'].apply(lambda x: 1 if x == 2.0 else 0)
monaco_copy['3_star'] = monaco_copy['stars'].apply(lambda x: 1 if x == 3.0 else 0)

In [ ]:
# Group by 'department' and sum 'bibs', '1_star', '2_star' and '3_star'
monaco_grouped = monaco_copy.groupby(['department_num', 'department'])[['green_stars', 'selected', 'bib_gourmand', '1_star', '2_star', '3_star']].sum()

In [ ]:
# Create a 'total_ stars' column - sum of stars
# Individual stars are summed here. ie, If 2 star then stars = 2
monaco_grouped['total_stars'] = monaco_grouped['1_star']*1 + monaco_grouped['2_star']*2 + monaco_grouped['3_star']*3

# Create a 'starred_restaurants' column = sum of starred restaurants
monaco_grouped['starred_restaurants'] =  monaco_grouped['1_star'] + monaco_grouped['2_star'] + monaco_grouped['3_star']

monaco_grouped['capital'] = 'Monaco'
monaco_grouped['region'] = "Provence-Alpes-Côte d'Azur"
print(f"Columns: {monaco_grouped.columns.tolist()}")

In [ ]:
# Reset index to turn 'department_num' and 'department' back into columns
monaco_grouped = monaco_grouped.reset_index()

# Add required demographic columns with placeholder 0.0 values
demographic_cols = [
    'GDP_millions(€)', 'GDP_per_capita(€)', 'poverty_rate(%)',
    'average_annual_unemployment_rate(%)', 'average_net_hourly_wage(€)',
    'municipal_population', 'population_density(inhabitants/sq_km)',
    'area(sq_km)'
]

for col in demographic_cols:
    monaco_grouped[col] = 0.0

# Reorder columns to match departmental_data
monaco_grouped = monaco_grouped[[
    'department_num', 'department', 'capital', 'region', 'selected',
    'bib_gourmand', '1_star', '2_star', '3_star', 'total_stars',
    'starred_restaurants', 'green_stars',
    'GDP_millions(€)', 'GDP_per_capita(€)', 'poverty_rate(%)',
    'average_annual_unemployment_rate(%)', 'average_net_hourly_wage(€)',
    'municipal_population', 'population_density(inhabitants/sq_km)',
    'area(sq_km)'
]]

monaco_grouped.head()

In [ ]:
monaco_grouped.info()

----
&nbsp;
### Adding individual restaurants

In [ ]:
# Create a separate DataFrame with star ratings, department, and coordinates
location_data_monaco = monaco[['stars', 'department_num', 'latitude', 'longitude']]

# Convert to a dictionary where keys are tuples of star rating and department
location_dict_monaco = (
    location_data_monaco
    .groupby(['stars', 'department_num'])[['latitude', 'longitude']]
    .apply(lambda df: list(zip(df.latitude, df.longitude)))
    .to_dict()
)

In [ ]:
# Create a mapping from star values to string labels
star_label_mapping = {
    0.25: 'Selected',
    0.5: 'Bib',
    1: '1',
    2: '2',
    3: '3'
}

# Create a function to map these dictionaries to original DataFrame for departments
def map_locations_department(row):
    return {star_label_mapping[stars]: location_dict_monaco.get((stars, row['department_num'])) for stars in [0.25, 0.5, 1, 2, 3]}

In [ ]:
# Apply this function to create a new 'locations' column
monaco_grouped['locations'] = monaco_grouped.apply(map_locations_department, axis=1)
monaco_grouped.head()

----
&nbsp;
### Merging with geoJSON data

In [ ]:
gdf_monaco = gpd.read_file(MONACO_GEOMETRY_PATH)
gdf_monaco.head()

In [ ]:
# Add the geometry column from gdf_monaco
monaco_grouped['geometry'] = gdf_monaco.geometry.values[0]

# Move geometry to second column
cols = monaco_grouped.columns.tolist()
cols.insert(1, cols.pop(cols.index('geometry')))
monaco_grouped = monaco_grouped[cols]

# Rename 'department_num' to 'code'
monaco_grouped = monaco_grouped.rename(columns={'department_num': 'code'})

# Convert to GeoDataFrame and set CRS (assuming WGS84 from original GeoJSON)
monaco_geo = gpd.GeoDataFrame(monaco_grouped, geometry='geometry', crs=gdf_monaco.crs)

# Preview
monaco_geo.head()

In [ ]:
monaco_geo.info()

In [ ]:
# Export the GeoDataFrame to a .geojson file
save_geojson(monaco_geo, "monaco_restaurants.geojson")